# Career Brain Training — Qwen3-1.7B QLoRA

Train a career avatar model using QLoRA on a Kaggle T4 GPU.
Training data: `dataset` path from the training config (`BRAIN_TRAIN_CONFIG`, default `data/brain/training_configs/tailor.yaml`; OpenAI chat format)
Base model: Qwen3-1.7B
Quantization: 4-bit NF4 + LoRA rank 8


In [ ]:
# Single-GPU training: T4x2 exposes two GPUs, which makes device_map="auto" shard
# the model and Trainer wrap it in DataParallel -> RuntimeError. One 16GB T4 is
# plenty for Qwen3-1.7B QLoRA.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # effective for fresh kernels/subprocesses

import torch
if torch.cuda.device_count() != 1:
    # Kernel already initialized CUDA with 2 GPUs (env var too late) - hide GPU 1
    # at runtime so device_map="auto" and Trainer see a single device.
    torch.cuda.device_count = lambda: 1
print("Visible GPUs:", torch.cuda.device_count())

# Install dependencies
!pip install -q transformers peft trl accelerate bitsandbytes datasets
!pip install -q sentencepiece protobuf

In [ ]:
# Load training data — the file is chosen by the training config
# (BRAIN_TRAIN_CONFIG env, default tailor.yaml), never by glob order: the old
# priority-glob could silently train on a stale v1 dump attached alongside the
# real dataset.
import glob
import json
import os
from pathlib import Path
import yaml
from datasets import Dataset, load_dataset

cfg_path = os.environ.get("BRAIN_TRAIN_CONFIG", "data/brain/training_configs/tailor.yaml")
cfg = yaml.safe_load(open(cfg_path))
want = Path(cfg["dataset"]).name  # e.g. tailor_pairs.jsonl / qa_pairs.jsonl

# Glob hits for the wanted file name (local cwd + Kaggle input mounts), plus
# the exact repo-relative path from the config when it exists (local runs).
candidates = sorted(set(
    glob.glob(want) + glob.glob(f"/kaggle/input/**/{want}", recursive=True)))
if os.path.exists(cfg["dataset"]):
    candidates = sorted(set(candidates) | {cfg["dataset"]})
if len(candidates) != 1:
    raise FileNotFoundError(
        f"BRAIN_TRAIN_CONFIG={cfg_path} wants '{want}' but found "
        f"{len(candidates)} match(es): {candidates}. Attach exactly one "
        "dataset via 'Add Input' (right sidebar) or fix the config path."
    )
data_path = candidates[0]
print(f"Loading {want} from: {data_path}")

training_data = []
with open(data_path, "r") as f:
    for line in f:
        if line.strip():
            training_data.append(json.loads(line))

print(f"Loaded {len(training_data)} training pairs")
print(f"Example: {training_data[0]}")

# Convert to Dataset
dataset = Dataset.from_list(training_data)
print(dataset)

In [ ]:
# Load model with 4-bit quantization
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, EarlyStoppingCallback

# Free any model left on the GPU from a previous (failed) run - makes re-runs safe
for _v in ("trainer", "model"):
    if _v in globals():
        del globals()[_v]
gc.collect()
torch.cuda.empty_cache()

model_name = "Qwen/Qwen3-1.7B"

# Configure 4-bit quantization
# NOTE: T4 (sm_75) has no native bfloat16 - use fp16 compute dtype, else the
# fp16 GradScaler fails on BF16 grads ("_amp_foreach... not implemented for BFloat16")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded: {model_name}")
print(f"compute dtype: {model.model.layers[0].self_attn.q_proj.compute_dtype}")  # expect torch.float16
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

In [ ]:
# Apply LoRA config — hyperparameters come from the training config (BRAIN_TRAIN_CONFIG env, default tailor.yaml)
import os, yaml
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

cfg_path = os.environ.get("BRAIN_TRAIN_CONFIG", "data/brain/training_configs/tailor.yaml")
cfg = yaml.safe_load(open(cfg_path))
lora_cfg = cfg["lora"]

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=lora_cfg["r"], lora_alpha=lora_cfg["lora_alpha"],
    target_modules=lora_cfg["target_modules"],
    lora_dropout=lora_cfg.get("lora_dropout", 0.05),
    bias="none", task_type=lora_cfg.get("task_type", "CAUSAL_LM"),
)
print("training config:", cfg_path, "| r =", lora_cfg["r"], "| targets =", len(lora_cfg["target_modules"]))

# Apply LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Train with SFTTrainer
import inspect
from trl import SFTTrainer, SFTConfig

# Hold out a validation split (fixed seed) for eval-loss early stopping.
# datasets-native train_test_split keeps HF Dataset objects - sklearn's
# train_test_split returns python row-dict lists and breaks TRL packing/trainer.
# Runs here because cfg is loaded in the LoRA cell above; dataset is already built.
_split = dataset.train_test_split(test_size=cfg["val_split"], seed=42)
dataset, val_dataset = _split["train"], _split["test"]
print(f"train={len(dataset)} val={len(val_dataset)}")

# Training arguments
# NOTE: no fp16/bf16 flag on purpose - the 4-bit base keeps non-quantized layers
# (embed/lm_head/norm) in bf16, so mixed-precision GradScaler crashes on bf16 grads
# ("_amp_foreach... not implemented for BFloat16"). Plain fp32 training of the
# LoRA params is stable and fast enough at this scale.
# NOTE: TRL renamed max_seq_length -> max_length; accept both for version tolerance.
tr = cfg["training"]
sft_kwargs = dict(
    output_dir="./career-brain-lora",
    num_train_epochs=tr["num_train_epochs"],
    per_device_train_batch_size=tr["per_device_train_batch_size"],
    gradient_accumulation_steps=tr["gradient_accumulation_steps"],
    learning_rate=tr["learning_rate"],
    warmup_ratio=tr.get("warmup_ratio", 0.03),
    lr_scheduler_type=cfg.get("lr_scheduler_type", "cosine"),
    neftune_noise_alpha=cfg.get("neftune_noise_alpha"),
    gradient_checkpointing=tr.get("gradient_checkpointing", True),
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy=tr.get("eval_strategy", "epoch"),
    load_best_model_at_end=tr.get("load_best_model_at_end", True),
    metric_for_best_model=tr.get("metric_for_best_model", "eval_loss"),
    report_to="none",
)
if "packing" in inspect.signature(SFTConfig).parameters:
    sft_kwargs["packing"] = tr.get("packing", True)
else:
    # packing handled via max_length truncation on this TRL version
    pass

_max_len = tr.get("max_seq_length", 2048)
try:
    training_args = SFTConfig(max_length=_max_len, **sft_kwargs)
except TypeError:
    # Older TRL/transformers: max_seq_length + evaluation_strategy
    _legacy = {("evaluation_strategy" if k == "eval_strategy" else k): v
               for k, v in sft_kwargs.items()}
    training_args = SFTConfig(max_seq_length=_max_len, **_legacy)

# Initialize trainer
# Dataset has a "messages" column - TRL applies the chat template automatically
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    eval_dataset=val_dataset,
    args=training_args,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=tr["early_stopping_patience"])],
)

# Train
print("Starting training...")
trainer.train()
print("Training complete!")

In [ ]:
# Save adapter
output_path = "./career-brain-lora-adapter"
model.save_pretrained(output_path)
tokenizer.save_pretrained(output_path)
print(f"LoRA adapter saved to: {output_path}")
print("\nNext steps:")
print("1. Download the adapter folder from Kaggle output")
print("2. Merge: python scripts/brain/merge_lora.py --adapter career-brain-lora-adapter --output career-brain-merged")
print("3. Convert: python scripts/brain/convert_to_gguf.py --input career-brain-merged --output career-brain.gguf")
print("4. Quantize: llama-quantize career-brain.gguf career-brain-q4.gguf Q4_K_M")
print("5. Import: python scripts/brain/import_to_ollama.py --gguf career-brain-q4.gguf")
